In [51]:
from langchain_community.chains.pebblo_retrieval.models import Prompt
!pip install -q langchain-google-genai langchain-experimental langchain-community langchain networkx langchain-core json-repair langgraph langchain_openai tavily-python graphviz matplotlib langchain_neo4j

In [52]:
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, ToolMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from tavily import TavilyClient
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from prompts import *

import os



In [53]:
load_dotenv()
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "intent-translation-main"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"


In [54]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)
llm_bigger = ChatOpenAI(model="gpt-4.1", temperature=0.8)
# llm_code = ChatOpenAI(model="gpt-4o",  temperature=0.8)


hasKG = True
run_name = "abstract/run1"

In [55]:
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_openai import ChatOpenAI
infrastructure_graph_name= "icontinuum"
infrastructure_graph = Neo4jGraph(database=infrastructure_graph_name, enhanced_schema=True)
infrastructure_kg_schema = infrastructure_graph.get_schema

expert_graph = Neo4jGraph(database="expert-kg-slo", enhanced_schema=True)
expert_kg_schema = expert_graph.get_schema

In [56]:
cypher_qa_chain_infrastructure = GraphCypherQAChain.from_llm(
    cypher_llm=ChatOpenAI(temperature=0.8, model_name="gpt-4o-mini"),
    qa_llm=ChatOpenAI(temperature=0.8, model_name="gpt-4o-mini"),
    schema=infrastructure_kg_schema,
    graph=infrastructure_graph,
    verbose=True,
    use_function_response=True,
    allow_dangerous_requests=True,
    validate_cypher = True,
    top_k=50
)

cypher_qa_chain_expert = GraphCypherQAChain.from_llm(
    cypher_llm=ChatOpenAI(temperature=0.8, model_name="gpt-4o-mini"),
    qa_llm=ChatOpenAI(temperature=0.8, model_name="gpt-4o-mini"),
    schema=expert_kg_schema,
    graph=expert_graph,
    verbose=True,
    use_function_response=True,
    allow_dangerous_requests=True,
    validate_cypher = True
)

In [57]:
@tool
def web_search(query):
  """
    Performs a web search using the Tavily API.
  """
  query = query.strip('"')

  try:
      # Initialize the client
      client = TavilyClient()

      # Perform the search
      response = client.search(
          query=query,
          search_depth="advanced",  # or "basic" depending on your needs
          max_results=5
      )
      return response
  except Exception as e:
      print(f"Search error: {e}")
      return None


In [58]:
@tool
def graphrag_infrastructure_search(query):
  """
    Performs a query on an Infrastructure Knowledge Graph (GraphRAG).
  """
  query = query.strip('"')

  try:
      # Perform the query
      response = cypher_qa_chain_infrastructure.invoke(query)
      return response
  except Exception as e:
      print(f"GraphRAG error: {e}")
      return None


@tool
def graphrag_expert_search(query):
  """
    Performs a query on an Expert Knowledge Graph (GraphRAG).
  """
  query = query.strip('"')

  try:
      # Perform the query
      response = cypher_qa_chain_expert.invoke(query)
      return response
  except Exception as e:
      print(f"GraphRAG error: {e}")
      return None

In [59]:
class ResearchState(TypedDict):
    messages: list
    iteration: int
    answer: str
    system_state: str
    infrastructure_kg_guidelines: str
    infrastructure_kg_queries: str
    query: str
    refined_query: str
    rl_output: str
    metrics: str


agent_builder = StateGraph(ResearchState)
MAX_ITERATIONS = 20

In [60]:
if hasKG:
    tools = [web_search, graphrag_expert_search, graphrag_infrastructure_search]
else:
    tools = [web_search]
tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm_bigger.bind_tools(tools)

In [61]:

from langchain_core.prompts import PromptTemplate


def refine_query(state: ResearchState) -> ResearchState:
    """Refine the user query."""
    user_message = state["messages"][-1].content
    print(user_message)

    state["query"] = user_message
    # Output parser you can now use
    prompt = PromptTemplate.from_template(get_query_refiner_prompt())
    input_prompt = prompt.format(query=user_message)
    response = llm.invoke([SystemMessage(content=input_prompt)])
    new_messages = state["messages"] + [response]
    state["messages"] = new_messages
    state["refined_query"] = response.content
    return state


In [62]:
def get_relevant_infrastructure(state: ResearchState) -> ResearchState:
    """Using the infrastructure KG schema decides on nodes and relationships relevant for the problem"""
    prompt = PromptTemplate.from_template(get_infrastructure_knowledge_prompt())
    input_prompt = prompt.format(schema = infrastructure_kg_schema)
    output_message = llm.invoke([SystemMessage(content=input_prompt)]+[HumanMessage(content=state["refined_query"])])
    state["infrastructure_kg_guidelines"] = output_message.content
    return state

def extract_infrastructure_kg_queries(state: ResearchState) -> ResearchState:
    """Using the infrastructure KG extraction guidelines generates queries for the KG"""
    prompt = PromptTemplate.from_template(extract_infrastructure_kg_queries_prompt())
    input_prompt = prompt.format(guidelines = state["infrastructure_kg_guidelines"], schema = infrastructure_kg_schema)
    output_message = llm.invoke([SystemMessage(content=input_prompt)])
    state["infrastructure_kg_queries"] = output_message.content
    return state

def execute_multiple_queries(state: ResearchState) -> ResearchState:
    """Execute multiple queries on the KG and get the results."""
    queries = state["infrastructure_kg_queries"].split("\n")
    results = []
    for query in queries:
        try:
            response = cypher_qa_chain_infrastructure.invoke(query)
        except Exception as e:
            print(f"Error executing query: {query}")
            print(e)
            continue
        results.append(response["result"])

    if "system_state" in state and state["system_state"]:
        state["system_state"] = state["system_state"] + "\n" + "\n".join(results)
    else:
        state["system_state"] = "\n".join(results)

    return state

def system_state_validation_node(state: ResearchState):
    prompt = PromptTemplate.from_template(get_state_validation_prompt())
    input_prompt = prompt.format(guidance = state["infrastructure_kg_guidelines"], system_state = state["system_state"], queries= state["infrastructure_kg_queries"])
    output_message = llm.invoke([SystemMessage(content=input_prompt)])
    new_messages = state["messages"] + [output_message]
    state["messages"] = new_messages
    if output_message.content != "OK":
        state["iteration"] += 1
        state["infrastructure_kg_queries"] = output_message.content
    else:
        state["iteration"] = 0
    return state

def should_correct_state(state: ResearchState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.content == "OK" or state["iteration"]>3:
        return "llm_call"
    else:
        return "execute_multiple_queries"


In [63]:
def llm_call(state: ResearchState):
    """LLM decides the next action."""
    if not hasKG:
        state["system_state"] = "No information"
        state["metrics"] = "No information"
    prompt = PromptTemplate.from_template(get_research_system_prompt())
    input_prompt = prompt.format(system_state = state["system_state"], intent=state["query"], refined_query = state["refined_query"])
    output_message = llm_with_tools.invoke([SystemMessage(content=input_prompt)] + state["messages"])

    new_messages = state["messages"] + [output_message]
    state["messages"] = new_messages

    answer = new_messages[-1].content
    state["answer"] = answer
    state["iteration"] += 1
    print(f"LLM Answer: {answer}")
    print(f'Iteration: {state["iteration"]+1}')

    return state

In [64]:
def tool_node(state: ResearchState):
    """Perform the tool calls."""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))

    print(f'Interation: {state["iteration"]}')
    print(f"Tool Message: {result}")


    # Update iteration count
    return {"messages": state["messages"] + result, "iteration": state["iteration"] + 1}

In [65]:
def answer_validation_node(state: ResearchState):
    """Validate the answer."""
    # Check if the answer is valid
    output_message = llm_with_tools.invoke([SystemMessage(content=get_answer_validation_prompt())]+[HumanMessage(content=state["answer"])])
    new_messages = state["messages"] + [output_message]
    state["messages"] = new_messages

    return state


In [66]:
def answer_parser_node(state: ResearchState):
    prompt = PromptTemplate.from_template(gymnasium_parser_prompt())
    input_prompt = prompt.format(query = state["query"], system_state = state["system_state"], answer = state["answer"])
    parsed_answer = llm_bigger.invoke([SystemMessage(content=input_prompt)])
    state["rl_output"] = parsed_answer.content

    return state

def rl_output_validation_node(state: ResearchState):
    """Validate the RL output."""
    # Check if the created python code needs improvements
    prompt = PromptTemplate.from_template(get_rl_output_validation_prompt())
    input_prompt = prompt.format(code = state["rl_output"])
    output_message = llm.invoke([SystemMessage(content=input_prompt)])
    prompt2 = PromptTemplate.from_template(rewrite_code_prompt())
    input_prompt2 = prompt2.format(code = state["rl_output"], suggestions = output_message.content)
    output_message2 = llm.invoke([SystemMessage(content=input_prompt2)])
    state["rl_output"] = output_message2.content
    return state

In [67]:
def should_continue(state: ResearchState):
    messages = state["messages"]
    last_message = messages[-1]
    count_iter = state["iteration"]
    if last_message.tool_calls and count_iter < MAX_ITERATIONS:
        return "tools"
    return "answer_validation_node"

In [68]:
def should_correct_answer(state: ResearchState):
    """If the answer needs correction, go to llm_call"""
    messages = state["messages"]
    last_message = messages[-1]
    count_iter = state["iteration"]
    if last_message.content == "OK" or count_iter > MAX_ITERATIONS :
        return "answer_parser_node"
    return "llm_call"

In [69]:
if hasKG:
    agent_builder.add_node("refine_query", refine_query)
    agent_builder.add_node("get_relevant_infrastructure", get_relevant_infrastructure)

    agent_builder.add_node("llm_call", llm_call)
    agent_builder.add_node("tools", tool_node)
    agent_builder.add_node("answer_parser_node", answer_parser_node)
    agent_builder.add_node("answer_validation_node", answer_validation_node)
    agent_builder.add_node("rl_output_validation_node", rl_output_validation_node)
    agent_builder.add_node("extract_infrastructure_kg_queries", extract_infrastructure_kg_queries)
    agent_builder.add_node("execute_multiple_queries", execute_multiple_queries)
    agent_builder.add_node("system_state_validation_node", system_state_validation_node)


    agent_builder.add_edge(START, "refine_query")
    agent_builder.add_edge("refine_query", "get_relevant_infrastructure")
    agent_builder.add_edge("get_relevant_infrastructure", "extract_infrastructure_kg_queries")
    agent_builder.add_edge("extract_infrastructure_kg_queries", "execute_multiple_queries")
    agent_builder.add_edge("execute_multiple_queries", "system_state_validation_node")

    agent_builder.add_conditional_edges("system_state_validation_node", should_correct_state, {"llm_call": "llm_call", "execute_multiple_queries": "execute_multiple_queries"})
    agent_builder.add_conditional_edges("llm_call", should_continue, {"tools": "tools", "answer_validation_node": "answer_validation_node"})
    agent_builder.add_edge("tools", "llm_call")
    agent_builder.add_conditional_edges("answer_validation_node", should_correct_answer, {"llm_call": "llm_call", "answer_parser_node": "answer_parser_node"})
    agent_builder.add_edge("answer_parser_node", "rl_output_validation_node")
    agent_builder.add_edge("rl_output_validation_node", END)
else:

    agent_builder.add_node("refine_query", refine_query)
    agent_builder.add_node("llm_call", llm_call)
    agent_builder.add_node("tools", tool_node)
    agent_builder.add_node("answer_parser_node", answer_parser_node)
    agent_builder.add_node("answer_validation_node", answer_validation_node)
    agent_builder.add_node("rl_output_validation_node", rl_output_validation_node)

    agent_builder.add_edge(START, "refine_query")
    agent_builder.add_edge("refine_query", "llm_call")
    agent_builder.add_conditional_edges("llm_call", should_continue, {"tools": "tools", "answer_validation_node": "answer_validation_node"})
    agent_builder.add_edge("tools", "llm_call")
    agent_builder.add_conditional_edges("answer_validation_node", should_correct_answer, {"llm_call": "llm_call", "answer_parser_node": "answer_parser_node"})
    agent_builder.add_edge("answer_parser_node", "rl_output_validation_node")
    agent_builder.add_edge("rl_output_validation_node", END)

agent = agent_builder.compile()

In [70]:
human_query = """
### Research and provide possible solutions for fixing this problem: "Because of the increased number of users, the image resizing service is taking too long to process images. I want to improve the total latency, but when there are fewer users, I want to avoid overprovisioning."

"""

# Ensure initial state includes "iteration"
state = {"messages": [HumanMessage(content=human_query)], "iteration": 0}
config = {"run_name":run_name}
# Invoke agent
final_state = agent.invoke(state, config)  # This returns a dictionary

In [71]:
# from IPython.display import Image, display
#
# try:
#     display(Image(agent.get_graph().draw_mermaid_png()))
# except Exception:
#     # This requires some extra dependencies and is optional
#     pass


In [72]:
def create_test_description(state, folder_name):
    # Get the user query
    user_query = state['query']

    # Infrastructure graph information
    infra_graph_name = infrastructure_graph_name if hasKG else "No KG used"

    # Extract model information directly from the chain objects
    cypher_llm_info = {
        "model": "gpt-4o-mini",
        "temperature": 0.8
    }

    qa_llm_info = {
        "model": "gpt-4o-mini",
        "temperature": 0.8
    }

    # Model information
    model_info = {
        "llm": {"model": llm.model_name, "temperature": llm.temperature},
        "llm_bigger": {"model": llm_bigger.model_name, "temperature": llm_bigger.temperature},
        "cypher_llm": cypher_llm_info,
        "qa_llm": qa_llm_info
    }

    # Create description text
    description = f"""User Query:
{user_query}

Infrastructure Information:
- Infrastructure Graph: {infra_graph_name}

Model Information:
- Primary LLM: {model_info['llm']['model']} (temperature: {model_info['llm']['temperature']})
- Secondary LLM: {model_info['llm_bigger']['model']} (temperature: {model_info['llm_bigger']['temperature']})
- Cypher LLM: {model_info['cypher_llm']['model']} (temperature: {model_info['cypher_llm']['temperature']})
- QA LLM: {model_info['qa_llm']['model']} (temperature: {model_info['qa_llm']['temperature']})
"""

    # Write to file
    with open(f"{folder_name}/test_description.txt", "w", encoding="utf-8") as f:
        f.write(description)

In [73]:
def write_result(run_name, final_state):
    """Writes the final state to files."""
    rf_q = final_state["refined_query"]
    folder_name = "outputs/resize/"+run_name
    os.makedirs(folder_name, exist_ok=True)

    with open(f"{folder_name}/rf_q.txt", "w", encoding="utf-8") as file:
        file.write(rf_q)

    text = final_state["answer"]
    with open(f"{folder_name}/answer.txt", "w", encoding="utf-8") as file:
        file.write(text)

    with open(f"{folder_name}/system_state.txt", "w", encoding="utf-8") as file:
        file.write(final_state["system_state"])

    with open(f"{folder_name}/output.py", "w", encoding="utf-8") as f:
        f.write(final_state['rl_output'])

#create_test_description(final_state, folder_name)

In [74]:
write_result(run_name, final_state)


### Research and provide possible solutions for fixing this problem: "Because of the increased number of users, the image resizing service is taking too long to process images. I want to improve the total latency, but when there are fewer users, I want to avoid overprovisioning."




> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (s:Service {name: 'Resize image'})-[:HAS_METRIC]->(m:Metric)
RETURN m

Full Context:
[{'m': {'unit': 'ms', 'tunable': True, 'name': 'Service Latency', 'value': 550}}, {'m': {'unit': 'cores', 'tunable': False, 'name': 'Service CPU Limit Cores', 'value': 0.25}}, {'m': {'unit': 'MiB', 'tunable': False, 'name': 'Service Memory Limit MiB', 'value': 512}}]

> Finished chain.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (s:Service {name: 'Resize image'})-[:HAS_METRIC]->(m:Metric {name: 'Service Latency'})
RETURN m.value AS latency, m.tunable AS isTunable

Full Context:
[{'latency': 550, 'isTunable': True}]

>